# Phase B — DeBERTa cross-encoder OOF features (Colab/GPU)

The torch slice runs here (CPU on the dev box is too slow). This notebook fine-tunes the
`microsoft/deberta-v3-base` cross-encoder and writes **out-of-fold** signals
(`neural_oof.parquet`) per dataset, row-aligned to `features.parquet` on the **same grouped
leave-questions-out folds** as the GBM (so no leakage).

**Inputs to upload** (zip `data/processed/<name>/encoder.parquet` + `features.parquet`, and the repo).
Set `Runtime → Change runtime type → GPU` first.

**After you download `neural_oof.zip`**, unzip into `data/processed/<name>/` locally, then run
(CPU, lightgbm installed):

```
make ablations        # now includes branch D (neural) → −D / only-D variants
make train2d          # hybrid tuned headline + cluster-bootstrap significance
python -m asag.models.neural_compare    # neural-only / feature-only / hybrid three-way table
```

That produces `reports/phase_hybrid/three_way.{json,md}` + the figure the paper's Table 3 uses.

In [ ]:
# 1. Setup — zero-conflict on current Colab (Python 3.12, torch 2.11, transformers 5.x).
#    We do NOT pip-install the project (its requires-python is <3.12 and pip would refuse;
#    the Arabic source path also breaks editable .pth files). Instead we put src/ on sys.path.
#    We do NOT reinstall/downgrade torch, transformers, numpy, etc. — every import the neural
#    slice needs is already on Colab. The ONLY missing package is loguru, so that's all we add.
!pip -q install loguru
# upload Explainable-Hybrid-ASAG.zip (code + data/processed parquets), then:
!unzip -q -o Explainable-Hybrid-ASAG.zip -d /content/
import sys, os
REPO = '/content/Explainable-Hybrid-ASAG'
sys.path.insert(0, os.path.join(REPO, 'src'))     # import asag from source — no pip, no requires-python clash
os.environ['ASAG_PROJECT_ROOT'] = REPO            # config.py walks here for pyproject + configs/data.yaml
import asag
from asag.config import load_data_config
print('asag loaded from', asag.__file__)
print('project root  ', os.environ['ASAG_PROJECT_ROOT'])

In [ ]:
# 1b. Fail-fast compatibility check (transformers 5.x / torch 2.11 / peft).
#     Verifies the exact APIs the neural slice uses load on THIS Colab before the long run.
import torch, transformers
print('python      ', sys.version.split()[0])
print('torch       ', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup  # stable across 4.x→5.x
import peft  # LoRA backend for small corpora
# deberta-v3 ships no tokenizer.json → fast tokenizer is built from SentencePiece via protobuf.
_tok = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')
_enc = _tok('a reference [SEP] b', 'student answer', truncation=True, max_length=16,
            padding='max_length', return_tensors='pt')
assert 'input_ids' in _enc and _enc['input_ids'].shape[1] == 16
print('peft        ', peft.__version__, '| deberta-v3 tokenizer OK, keys:', list(_enc.keys()))
print('✓ no conflicts — APIs resolve on this runtime')

In [ ]:
# 2. configure the run (LoRA + dev-selection for the small corpora)
from asag.config import load_data_config
cfg = load_data_config()
cfg.neural.device = 'cuda'
cfg.neural.epochs = 4
cfg.neural.lora_enabled = True      # train <1% of params; protects Mohler (616) / SAF (1.7k)
cfg.neural.select_on_dev = True
cfg.neural.seeds = [42]
# ordinal head (asap_sas, mindreading) is global; default 'corn'. To try CORAL instead:
#   cfg.neural.ordinal_head = 'coral'
print(cfg.neural)

In [ ]:
# 2. configure the run (LoRA + dev-selection for the small corpora)
from asag.config import load_data_config
cfg = load_data_config()
cfg.neural.device = 'cuda'
cfg.neural.epochs = 4
cfg.neural.lora_enabled = True      # train <1% of params; protects Mohler (616) / SAF (1.7k)
cfg.neural.select_on_dev = True
cfg.neural.seeds = [42]
print(cfg.neural)

In [ ]:
# 4. sanity: OOF coverage per dataset (no lightgbm needed on Colab).
#    Coverage should be ~100% — every row gets an out-of-fold prediction. Anything lower
#    means a fold/split row was left uncovered (investigate before trusting the hybrid).
#    The three-way comparison (neural-only / feature-only / hybrid) + significance run
#    LOCALLY after download — Colab only produces the OOF signals.
import pandas as pd, numpy as np
for n in _present:
    p = _proc / n / 'neural_oof.parquet'
    if p.exists():
        d = pd.read_parquet(p)
        print(f'{n:14s} rows={len(d):6d}  OOF coverage={np.isfinite(d["neural_score"]).mean():.1%}')
    else:
        print(f'{n:14s} no neural_oof.parquet (extraction skipped or failed)')

In [ ]:
# 4. (optional) verify the hybrid lifts SAF test_uq before downloading
from asag.models.evaluate import evaluate_dataset
r = evaluate_dataset('saf', cfg)
print('SAF test_uq pearson (hybrid):', r['headline']['gbm'].get('mean'))

In [ ]:
# 5. zip the neural_oof parquets to download back into data/processed/<name>/
!cd /content/Explainable-Hybrid-ASAG && zip -q -r /content/neural_oof.zip data/processed/*/neural_oof.parquet
from google.colab import files; files.download('/content/neural_oof.zip')